In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

CFG = {
    'D': 256,
    'H': 4,
    'N': 32768,
    'TOP_K': 8,
    'VOCAB': 50257,
    'RETRIEVAL_TOP_K': 3,
    'EPOCHS': 15,          
    'DEVICE': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'RAG_MODEL': 'all-MiniLM-L6-v2',
    'MAX_LEN': 100
}

print(f"Running on {CFG['DEVICE']}")
def find_file(filename_part):
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if filename_part in name: return os.path.join(root, name)
    if os.path.exists(filename_part): return filename_part
    return None

train_path = find_file('train (2).csv') or find_file('train.csv')
test_path = find_file('test (2).csv') or find_file('test.csv')
novels = [find_file('The Count of Monte Cristo.txt'), find_file('In search of the castaways.txt')]
novels = [n for n in novels if n]

class DenseRetriever:
    def __init__(self, novel_paths, model_name=CFG['RAG_MODEL']):
        self.chunks = []
        self.source_map = []
        
        print(f"Loading Dense Model ({model_name})...")
        self.model = SentenceTransformer(model_name, device=str(CFG['DEVICE']))
        
        print("Indexing Novels for Semantic Search...")
        for path in novel_paths:
            with open(path, 'r', encoding='utf-8') as f: text = f.read()
            words = text.split()
            chunk_size = 120
            for i in range(0, len(words), chunk_size):
                chunk = " ".join(words[i:i+chunk_size])
                self.chunks.append(chunk)
                self.source_map.append(os.path.basename(path))

        print(f"Encoding {len(self.chunks)} chunks (this may take a moment)...")
        self.chunk_embeddings = self.model.encode(self.chunks, convert_to_tensor=True, show_progress_bar=True)
        print("Indexing Complete.")

    def retrieve(self, query, top_k=3):
        query_vec = self.model.encode(query, convert_to_tensor=True)
        scores = torch.cosine_similarity(query_vec.unsqueeze(0), self.chunk_embeddings)
        top_k_scores, top_indices = torch.topk(scores, k=top_k)
        
        return [self.chunks[i] for i in top_indices.cpu().numpy()]

retriever = DenseRetriever(novels)

class BDH_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(CFG['VOCAB'], CFG['D'])
        self.ln = nn.LayerNorm(CFG['D'])
        self.decoder = nn.Parameter(torch.randn(CFG['H'], CFG['D'], CFG['N'] // CFG['H']) * 0.02)

    def forward(self, idx):
        x = self.ln(self.wte(idx))
        logits = torch.einsum('btd,hdn->bhtn', x, self.decoder)
        topk_vals, topk_indices = torch.topk(logits, k=CFG['TOP_K'], dim=-1)
        
        return x, topk_indices

class DualStreamMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.global_state = nn.Parameter(torch.randn(CFG['N'], CFG['D']) * 0.01)
        self.gate_net = nn.Sequential(nn.Linear(CFG['D']*2, CFG['D']), nn.ReLU(), nn.Linear(CFG['D'], 1), nn.Sigmoid())
        self.forget_net = nn.Sequential(nn.Linear(CFG['D']*2, 1), nn.Sigmoid())
        self.recon_head = nn.Linear(CFG['D'], CFG['D'])
        self.wm_proj = nn.Linear(CFG['D'], CFG['D'])

    def update_global(self, attr_vec, active_indices):
        flat_indices = active_indices.flatten().unique()
        current_memory_rows = self.global_state[flat_indices] 
        fact_expanded = attr_vec.expand(len(flat_indices), -1)
        
        combined = torch.cat([fact_expanded, current_memory_rows], dim=-1)
        i_gate = self.gate_net(combined) 
        f_gate = self.forget_net(combined)
        
        new_memory_rows = (f_gate * current_memory_rows) + (i_gate * fact_expanded)
        
        return new_memory_rows, flat_indices, i_gate, self.recon_head(new_memory_rows)

    def probe(self, backstory_vec, active_indices, retrieved_vecs):
        flat_indices = active_indices.flatten().unique()
        relevant_memory = self.global_state[flat_indices]
        memory_trace = torch.mean(relevant_memory, dim=0, keepdim=True)
        global_trace = self.recon_head(memory_trace)
        sim_global = F.cosine_similarity(global_trace, backstory_vec)
        if retrieved_vecs is not None and len(retrieved_vecs) > 0:
            wm_trace = self.wm_proj(torch.mean(retrieved_vecs, dim=0, keepdim=True))
            sim_local = F.cosine_similarity(wm_trace, backstory_vec)
        else:
            sim_local = sim_global 
            
        return sim_global, sim_local

def validate(model_enc, model_mem, val_df, retriever):
    model_enc.eval(); model_mem.eval()
    scores, labels = [], []
    label_map = {'consistent': 1, 'contradict': 0}
    
    with torch.no_grad():
        for _, row in val_df.iterrows():
            b_words = str(row['content']).split()
            if not b_words: continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec_seq, b_indices = model_enc(b_toks)
            b_vec = torch.mean(b_vec_seq, dim=1).squeeze(0)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=2)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv, _ = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            sim_g, sim_l = model_mem.probe(b_vec.unsqueeze(0), b_indices, ret_tensor)
            score = (0.4 * sim_g) + (0.6 * sim_l)
            scores.append(score.item())
            labels.append(label_map[row['label']])
    
    best_f1 = 0
    best_acc = 0
    for t in np.linspace(-0.5, 0.5, 50):
        preds = [1 if s >= t else 0 for s in scores]
        f1 = f1_score(labels, preds, average='weighted')
        if f1 > best_f1: 
            best_f1 = f1
            best_acc = balanced_accuracy_score(labels, preds)
            
    return best_f1, best_acc

def train_hybrid_dense(model_enc, model_mem, optimizer, train_df, val_df, novels):
    print(f"Starting Dense Hybrid Training (Full {CFG['EPOCHS']} Epochs)...")
    
    best_val_f1 = -1
    
    for epoch in range(CFG['EPOCHS']):
        model_enc.train(); model_mem.train()
        
        for novel in novels:
            with open(novel, 'r', encoding='utf-8') as f: text = f.read()
            chapters = re.split(r'(?im)^\s*Chapter\s+(?:[0-9]+|[IVXLCDM]+)\b', text)
            chapters = [c for c in chapters if len(c) > 500]
            
            for chapter in chapters[:15]: 
                words = chapter.split()[:100]
                for word in words:
                    optimizer.zero_grad()
                    tid = torch.tensor([[hash(word) % CFG['VOCAB']]], device=CFG['DEVICE'])
                    
                    vec_seq, indices = model_enc(tid) 
                    vec = vec_seq.squeeze(0).squeeze(0)
                    new_rows, active_idx, gate_val, recon = model_mem.update_global(vec, indices)
                    
                    model_mem.global_state.data[active_idx] = new_rows.detach()
                    loss = F.mse_loss(recon, vec.expand_as(recon).detach()) + 0.1*torch.mean(gate_val)
                    loss.backward()
                    optimizer.step()

        total_cons_loss = 0
        batch = train_df.sample(min(64, len(train_df)))
        optimizer.zero_grad()
        
        for _, row in batch.iterrows():
            b_words = str(row['content']).split()
            if not b_words: continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec_seq, b_indices = model_enc(b_toks)
            b_vec = torch.mean(b_vec_seq, dim=1).squeeze(0)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=2)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv, _ = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            sim_g, sim_l = model_mem.probe(b_vec.unsqueeze(0), b_indices, ret_tensor)
            hybrid_sim = (sim_g + sim_l) / 2.0
            
            target = 1.0 if row['label'] == 'consistent' else -1.0
            loss = F.relu(0.5 - hybrid_sim) if target == 1 else F.relu(hybrid_sim + 0.5)
            total_cons_loss += loss

        total_cons_loss = total_cons_loss / len(batch)
        total_cons_loss.backward()
        optimizer.step()
        val_f1, val_acc = validate(model_enc, model_mem, val_df, retriever)
        print(f"Epoch {epoch+1:02d} | Loss: {total_cons_loss.item():.4f} | Val F1: {val_f1:.4f} | Val Bal Acc: {val_acc:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({'enc': model_enc.state_dict(), 'mem': model_mem.state_dict()}, "best_hybrid_model.pth")
            print("  >>> New Best Model Saved!")

def predict_dataset(model_enc, model_mem, df, is_test=True):
    print("Loading Best Model...")
    checkpoint = torch.load("best_hybrid_model.pth", map_location=CFG['DEVICE'])
    model_enc.load_state_dict(checkpoint['enc'])
    model_mem.load_state_dict(checkpoint['mem'])
    
    model_enc.eval(); model_mem.eval()
    preds, ids = [], []
    THRESHOLD = 0.05 
    
    print(f"Generating Predictions...")
    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            ids.append(row['id'] if 'id' in row else idx)
            b_words = str(row['content']).split()
            if not b_words: 
                preds.append("consistent"); continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec_seq, b_indices = model_enc(b_toks)
            b_vec = torch.mean(b_vec_seq, dim=1).squeeze(0)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=3)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv, _ = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            sim_g, sim_l = model_mem.probe(b_vec.unsqueeze(0), b_indices, ret_tensor)
            score = (0.4 * sim_g) + (0.6 * sim_l)
            
            if score < THRESHOLD: preds.append("contradict")
            else: preds.append("consistent")
            
    if is_test: return pd.DataFrame({'id': ids, 'prediction': preds})
    else: return pd.DataFrame({'id': ids, 'char': df['char'], 'content': df['content'], 'true_label': df['label'], 'predicted_label': preds})

encoder = BDH_Encoder().to(CFG['DEVICE'])
memory = DualStreamMemory().to(CFG['DEVICE'])
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(memory.parameters()), lr=1e-3)

df_full = pd.read_csv(train_path)
df_train, df_val = train_test_split(df_full, test_size=0.2, random_state=42, stratify=df_full['label'])
df_test = pd.read_csv(test_path)

print(f"Train Size: {len(df_train)} | Val Size: {len(df_val)}")

train_hybrid_dense(encoder, memory, optimizer, df_train, df_val, novels)

val_preds = predict_dataset(encoder, memory, df_val, is_test=False)
val_preds.to_csv("validation_predictions.csv", index=False)
print("Saved validation_predictions.csv")

test_preds = predict_dataset(encoder, memory, df_test, is_test=True)
test_preds.to_csv("results.csv", index=False)
print("Saved results.csv")
print(test_preds.head())